# Federated Queries: Neo4j + Databricks Lakehouse

This notebook demonstrates federated queries that combine graph data in Neo4j with tabular data in Databricks Delta tables.

**Data split:**
- **Neo4j** (graph-native): Companies, Products, Competitors, Risk Factors — relationships and traversals
- **Databricks Delta** (tabular): Companies, Financial Metrics, Asset Managers, Holdings — aggregations and joins

**Prerequisites:**
- Run `01-simple-connect-test.ipynb` first to validate Neo4j connectivity and create the UC JDBC connection
- Upload the Neo4j Unity Catalog Connector JAR to a UC Volume
- Install the connector JAR as a cluster library

---

## Configuration

Update these values for your environment. Neo4j credentials should match notebook 1.

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these two sections only
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://123456.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
# If your catalog name contains hyphens (e.g., "uc-w-neo4j"), Spark SQL requires
# backtick quoting. Without it you'll get:
#   [INVALID_IDENTIFIER] The unquoted identifier uc-w-neo4j is invalid
#   and must be back quoted as: `uc-w-neo4j`.
# FQN handles this automatically.
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
UC_CONNECTION_NAME = f"`{UC_CATALOG}`.`{UC_SCHEMA}`.neo4j_jdbc"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Load Data into Databricks Lakehouse

Create a schema and volume in Unity Catalog, copy CSV files, and create Delta tables for financial data that will be queried alongside Neo4j graph data.

**Delta tables created:**
- `companies` — 6 public companies (bridge table, also in Neo4j)
- `financial_metrics` — 90 financial metrics extracted from 10-K filings
- `asset_managers` — 15 institutional investors
- `asset_manager_holdings` — 72 ownership records (shares held per company)

In [ ]:
# Create schema, volume, and copy CSV files
import os
import shutil

print("=" * 60)
print("SETUP: Creating Schema, Volume, and Copying Data")
print("=" * 60)

# Create schema (backtick-quoted to handle hyphens in catalog names)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
print(f"\n[OK] Schema: {FQN}")

# Create volume
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.{UC_VOLUME}")
print(f"[OK] Volume: {VOLUME_PATH}")

# Copy CSV files from workspace to volume
data_files = [
    "companies.csv",
    "financial_metrics.csv",
    "asset_managers.csv",
    "asset_manager_holdings.csv",
]

# Files are in the data/ directory relative to this notebook
notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
workspace_data_dir = f"/Workspace{notebook_dir}/data"

for filename in data_files:
    src = os.path.join(workspace_data_dir, filename)
    dst = os.path.join(VOLUME_PATH, filename)
    shutil.copy(src, dst)
    print(f"[OK] Copied {filename} -> {VOLUME_PATH}/")

print("\nStatus: PASS")

In [ ]:
# Create Delta tables from CSV files
print("=" * 60)
print("SETUP: Creating Delta Tables")
print("=" * 60)

tables = {
    "companies": "companies.csv",
    "financial_metrics": "financial_metrics.csv",
    "asset_managers": "asset_managers.csv",
    "asset_manager_holdings": "asset_manager_holdings.csv",
}

for table_name, csv_file in tables.items():
    table_fqn = f"{FQN}.{table_name}"
    spark.sql(f"DROP TABLE IF EXISTS {table_fqn}")
    spark.sql(f"""
        CREATE TABLE {table_fqn} AS
        SELECT * EXCEPT (_rescued_data)
        FROM read_files('{VOLUME_PATH}/{csv_file}',
            format => 'csv',
            header => true,
            inferColumnTypes => true)
    """)
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {table_fqn}").collect()[0]["cnt"]
    print(f"[OK] {table_fqn}: {count} rows")

print("\nStatus: PASS")

In [ ]:
# Verify Delta tables
print("=" * 60)
print("VERIFY: Delta Tables")
print("=" * 60)

print("\n--- Companies ---")
spark.sql(f"SELECT companyId, name, ticker FROM {FQN}.companies ORDER BY companyId").show(truncate=False)

print("--- Financial Metrics (sample) ---")
spark.sql(f"""
    SELECT fm.companyId, c.name AS company, fm.name AS metric, fm.value, fm.period
    FROM {FQN}.financial_metrics fm
    JOIN {FQN}.companies c ON fm.companyId = c.companyId
    ORDER BY fm.companyId, fm.metricId
    LIMIT 10
""").show(truncate=False)

print("--- Asset Manager Holdings (top 5 by shares) ---")
spark.sql(f"""
    SELECT am.name AS manager, c.name AS company, h.shares
    FROM {FQN}.asset_manager_holdings h
    JOIN {FQN}.asset_managers am ON h.managerId = am.managerId
    JOIN {FQN}.companies c ON h.companyId = c.companyId
    ORDER BY h.shares DESC
    LIMIT 5
""").show(truncate=False)

print("Status: PASS")

---

## Section 2: Load Graph Data into Neo4j

Create a knowledge graph of companies, their products, competitors, and risk factors. This data is graph-native — the value comes from traversing relationships, not aggregating rows.

**Graph structure:**
- `(:Company)-[:OFFERS]->(:Product)` — 6 companies, 30 products
- `(:Company)-[:COMPETES_WITH]->(:Company)` — 78 competitive relationships
- `(:Company)-[:HAS_RISK]->(:RiskFactor)` — 48 risk factors from SEC 10-K filings

In [ ]:
# Load graph data into Neo4j
from neo4j import GraphDatabase

print("=" * 60)
print("SETUP: Loading Graph Data into Neo4j")
print("=" * 60)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    with driver.session() as session:
        # Clear existing data
        session.run("MATCH (n) WHERE n:Company OR n:Product OR n:RiskFactor DETACH DELETE n")
        print("\n[OK] Cleared existing Company/Product/RiskFactor nodes")

        # ---------------------------------------------------------------
        # Company nodes (same 6 as in Delta — the bridge between worlds)
        # ---------------------------------------------------------------
        companies = [
            {"companyId": "C001", "name": "Amazon.com, Inc.", "ticker": "AMZN"},
            {"companyId": "C002", "name": "Apple Inc.", "ticker": "AAPL"},
            {"companyId": "C003", "name": "Microsoft Corporation", "ticker": "MSFT"},
            {"companyId": "C004", "name": "NVIDIA Corporation", "ticker": "NVDA"},
            {"companyId": "C005", "name": "PG&E Corporation", "ticker": "PCG"},
            {"companyId": "C006", "name": "PayPal Holdings, Inc.", "ticker": "PYPL"},
        ]
        session.run(
            "UNWIND $companies AS c CREATE (:Company {companyId: c.companyId, name: c.name, ticker: c.ticker})",
            companies=companies
        )
        print(f"[OK] Created {len(companies)} Company nodes")

        # ---------------------------------------------------------------
        # Product nodes + OFFERS relationships (5 per company)
        # ---------------------------------------------------------------
        products = [
            {"productId": "P001", "name": "A-to-z Guarantee", "companyId": "C001"},
            {"productId": "P011", "name": "AWS", "companyId": "C001"},
            {"productId": "P018", "name": "Amazon Prime", "companyId": "C001"},
            {"productId": "P019", "name": "Amazon Stores", "companyId": "C001"},
            {"productId": "P045", "name": "Blink", "companyId": "C001"},
            {"productId": "P015", "name": "AirPods", "companyId": "C002"},
            {"productId": "P020", "name": "App Store", "companyId": "C002"},
            {"productId": "P021", "name": "Apple Arcade", "companyId": "C002"},
            {"productId": "P022", "name": "Apple Card", "companyId": "C002"},
            {"productId": "P030", "name": "Apple Vision Pro", "companyId": "C002"},
            {"productId": "P009", "name": "AI Services", "companyId": "C003"},
            {"productId": "P010", "name": "AI-powered Productivity Services", "companyId": "C003"},
            {"productId": "P036", "name": "Azure", "companyId": "C003"},
            {"productId": "P037", "name": "Azure AI", "companyId": "C003"},
            {"productId": "P038", "name": "Azure AI Platform", "companyId": "C003"},
            {"productId": "P002", "name": "A100 Integrated Circuit", "companyId": "C004"},
            {"productId": "P003", "name": "A100X", "companyId": "C004"},
            {"productId": "P004", "name": "A800", "companyId": "C004"},
            {"productId": "P005", "name": "AGX", "companyId": "C004"},
            {"productId": "P006", "name": "AI Cloud Services", "companyId": "C004"},
            {"productId": "P050", "name": "Bundled Customer Electricity Supply", "companyId": "C005"},
            {"productId": "P051", "name": "Bundled Electric Service", "companyId": "C005"},
            {"productId": "P052", "name": "Bundled Gas Sales", "companyId": "C005"},
            {"productId": "P082", "name": "Diablo Canyon Nuclear Power Plant", "companyId": "C005"},
            {"productId": "P083", "name": "Diablo Canyon Unit 1", "companyId": "C005"},
            {"productId": "P046", "name": "Braintree", "companyId": "C006"},
            {"productId": "P047", "name": "Braintree (Unbranded Card Processing)", "companyId": "C006"},
            {"productId": "P048", "name": "Branded Credit Card", "companyId": "C006"},
            {"productId": "P049", "name": "Branded Debit Card", "companyId": "C006"},
            {"productId": "P053", "name": "Buy Now, Pay Later", "companyId": "C006"},
        ]
        session.run("""
            UNWIND $products AS p
            CREATE (prod:Product {productId: p.productId, name: p.name})
            WITH prod, p
            MATCH (c:Company {companyId: p.companyId})
            CREATE (c)-[:OFFERS]->(prod)
        """, products=products)
        print(f"[OK] Created {len(products)} Product nodes with OFFERS relationships")

        # ---------------------------------------------------------------
        # Competitor relationships
        # Some targets are internal (one of our 6), others are external
        # ---------------------------------------------------------------
        competitors = [
            {"source": "C001", "targetId": None, "targetName": "Alibaba Group"},
            {"source": "C001", "targetId": None, "targetName": "Alphabet"},
            {"source": "C001", "targetId": "C003", "targetName": "Microsoft Corporation"},
            {"source": "C001", "targetId": None, "targetName": "Shopify Inc."},
            {"source": "C001", "targetId": None, "targetName": "Walmart Inc."},
            {"source": "C002", "targetId": None, "targetName": "Google"},
            {"source": "C002", "targetId": "C003", "targetName": "Microsoft Corporation"},
            {"source": "C002", "targetId": None, "targetName": "Nintendo"},
            {"source": "C002", "targetId": None, "targetName": "Samsung Electronics Co. Ltd"},
            {"source": "C002", "targetId": None, "targetName": "Sony"},
            {"source": "C003", "targetId": None, "targetName": "Adobe"},
            {"source": "C003", "targetId": "C001", "targetName": "Amazon.com, Inc."},
            {"source": "C003", "targetId": "C002", "targetName": "Apple Inc."},
            {"source": "C003", "targetId": None, "targetName": "BMC"},
            {"source": "C003", "targetId": None, "targetName": "CA Technologies"},
            {"source": "C003", "targetId": None, "targetName": "Cisco Systems, Inc."},
            {"source": "C003", "targetId": None, "targetName": "Dell"},
            {"source": "C003", "targetId": None, "targetName": "Google"},
            {"source": "C003", "targetId": None, "targetName": "Hewlett-Packard"},
            {"source": "C003", "targetId": None, "targetName": "IBM"},
            {"source": "C003", "targetId": None, "targetName": "Intel Security Group"},
            {"source": "C003", "targetId": None, "targetName": "Lenovo"},
            {"source": "C003", "targetId": None, "targetName": "McAfee, LLC"},
            {"source": "C003", "targetId": None, "targetName": "Meta"},
            {"source": "C003", "targetId": None, "targetName": "Netflix, Inc."},
            {"source": "C003", "targetId": None, "targetName": "Nintendo"},
            {"source": "C003", "targetId": None, "targetName": "Nokia"},
            {"source": "C003", "targetId": None, "targetName": "Okta"},
            {"source": "C003", "targetId": None, "targetName": "OpenAI"},
            {"source": "C003", "targetId": None, "targetName": "Oracle"},
            {"source": "C003", "targetId": None, "targetName": "Proofpoint"},
            {"source": "C003", "targetId": None, "targetName": "Red Hat"},
            {"source": "C003", "targetId": None, "targetName": "SAP"},
            {"source": "C003", "targetId": None, "targetName": "Salesforce"},
            {"source": "C003", "targetId": None, "targetName": "Slack"},
            {"source": "C003", "targetId": None, "targetName": "Snowflake"},
            {"source": "C003", "targetId": None, "targetName": "Sony"},
            {"source": "C003", "targetId": None, "targetName": "Symantec"},
            {"source": "C003", "targetId": None, "targetName": "Tencent"},
            {"source": "C003", "targetId": None, "targetName": "VMware"},
            {"source": "C003", "targetId": None, "targetName": "Yahoo"},
            {"source": "C003", "targetId": None, "targetName": "Zoom"},
            {"source": "C004", "targetId": None, "targetName": "Advanced Micro Devices, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Advantest America Inc."},
            {"source": "C004", "targetId": None, "targetName": "Alibaba Group"},
            {"source": "C004", "targetId": None, "targetName": "Alphabet"},
            {"source": "C004", "targetId": "C001", "targetName": "Amazon.com, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Ambarella, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Arista Networks"},
            {"source": "C004", "targetId": None, "targetName": "Arm"},
            {"source": "C004", "targetId": None, "targetName": "BYD Auto"},
            {"source": "C004", "targetId": None, "targetName": "Baidu, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Broadcom Inc."},
            {"source": "C004", "targetId": None, "targetName": "Cisco Systems, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Hewlett Packard Enterprise Company"},
            {"source": "C004", "targetId": None, "targetName": "Hewlett-Packard"},
            {"source": "C004", "targetId": None, "targetName": "Intel Corporation"},
            {"source": "C004", "targetId": None, "targetName": "Juniper Networks, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Marvell Technology Group"},
            {"source": "C004", "targetId": None, "targetName": "Micron Technology"},
            {"source": "C004", "targetId": "C003", "targetName": "Microsoft Corporation"},
            {"source": "C004", "targetId": None, "targetName": "Qualcomm Incorporated"},
            {"source": "C004", "targetId": None, "targetName": "Quantum Corp."},
            {"source": "C004", "targetId": None, "targetName": "Renesas Electronics Corporation"},
            {"source": "C004", "targetId": None, "targetName": "SK Hynix"},
            {"source": "C004", "targetId": None, "targetName": "Samsung Electronics Co. Ltd"},
            {"source": "C004", "targetId": None, "targetName": "SoftBank Group Corp."},
            {"source": "C004", "targetId": None, "targetName": "Tesla, Inc."},
            {"source": "C004", "targetId": None, "targetName": "Texas Instruments Incorporated"},
            {"source": "C005", "targetId": None, "targetName": "CAISO"},
            {"source": "C005", "targetId": None, "targetName": "Pacific Power"},
            {"source": "C005", "targetId": None, "targetName": "San Diego Gas & Electric Company"},
            {"source": "C005", "targetId": None, "targetName": "Sempra Energy"},
            {"source": "C005", "targetId": None, "targetName": "Southern California Edison"},
            {"source": "C006", "targetId": None, "targetName": "Adyen N.V."},
            {"source": "C006", "targetId": None, "targetName": "Block, Inc."},
            {"source": "C006", "targetId": None, "targetName": "Stripe Inc."},
            {"source": "C006", "targetId": None, "targetName": "eBay"},
        ]

        # For internal competitors, link to existing Company nodes
        # For external competitors, create new Company nodes (name only)
        session.run("""
            UNWIND $competitors AS comp
            MATCH (source:Company {companyId: comp.source})
            CALL (source, comp) {
                WITH source, comp
                WHERE comp.targetId IS NOT NULL
                MATCH (target:Company {companyId: comp.targetId})
                CREATE (source)-[:COMPETES_WITH]->(target)
                UNION
                WITH source, comp
                WHERE comp.targetId IS NULL
                MERGE (target:Company {name: comp.targetName})
                CREATE (source)-[:COMPETES_WITH]->(target)
            }
        """, competitors=competitors)
        print(f"[OK] Created {len(competitors)} COMPETES_WITH relationships")

        # ---------------------------------------------------------------
        # Risk Factor nodes + HAS_RISK relationships (8 per company)
        # ---------------------------------------------------------------
        risk_factors = [
            {"riskId": "R006", "name": "A-to-z Guarantee Cost Risk", "companyId": "C001"},
            {"riskId": "R016", "name": "AWS Revenue Growth Impact", "companyId": "C001"},
            {"riskId": "R023", "name": "Acquisition and Investment Risk", "companyId": "C001"},
            {"riskId": "R024", "name": "Acquisition and Merger Risks", "companyId": "C001"},
            {"riskId": "R026", "name": "Additional Tax Liabilities and Collection Obligations", "companyId": "C001"},
            {"riskId": "R051", "name": "COVID-19 Pandemic Uncertainty", "companyId": "C001"},
            {"riskId": "R080", "name": "Claims, Litigation, and Government Investigations", "companyId": "C001"},
            {"riskId": "R094", "name": "Commercial Agreements and Strategic Alliances Risk", "companyId": "C001"},
            {"riskId": "R036", "name": "Antitrust Investigation Risk", "companyId": "C002"},
            {"riskId": "R037", "name": "App Store Regulatory Risk", "companyId": "C002"},
            {"riskId": "R043", "name": "Business Interruption", "companyId": "C002"},
            {"riskId": "R064", "name": "Carrier and Reseller Dependency Risk", "companyId": "C002"},
            {"riskId": "R118", "name": "Component Availability Risk", "companyId": "C002"},
            {"riskId": "R119", "name": "Component Shortage", "companyId": "C002"},
            {"riskId": "R121", "name": "Component Supply and Pricing Risk", "companyId": "C002"},
            {"riskId": "R140", "name": "Credit Risk", "companyId": "C002"},
            {"riskId": "R008", "name": "AI Data Usage Regulatory Risk", "companyId": "C003"},
            {"riskId": "R009", "name": "AI Development and Use Risk", "companyId": "C003"},
            {"riskId": "R011", "name": "AI Harm and Liability", "companyId": "C003"},
            {"riskId": "R012", "name": "AI Market Competition", "companyId": "C003"},
            {"riskId": "R013", "name": "AI Regulation Risk", "companyId": "C003"},
            {"riskId": "R015", "name": "AI Regulatory and Legal Liability", "companyId": "C003"},
            {"riskId": "R017", "name": "Abuse of Platforms", "companyId": "C003"},
            {"riskId": "R025", "name": "Acquisitions and Strategic Alliances Risk", "companyId": "C003"},
            {"riskId": "R007", "name": "AI Cloud Services Risk", "companyId": "C004"},
            {"riskId": "R010", "name": "AI Ethics and Regulation Risk", "companyId": "C004"},
            {"riskId": "R013b", "name": "AI Regulation Risk", "companyId": "C004"},
            {"riskId": "R014", "name": "AI Regulatory Restriction Risk", "companyId": "C004"},
            {"riskId": "R019", "name": "Acquisition Financing Risk", "companyId": "C004"},
            {"riskId": "R020", "name": "Acquisition Integration Risk", "companyId": "C004"},
            {"riskId": "R021", "name": "Acquisition Regulatory Risk", "companyId": "C004"},
            {"riskId": "R022", "name": "Acquisition Termination Risk", "companyId": "C004"},
            {"riskId": "R001", "name": "2017 Northern California Wildfire Claims", "companyId": "C005"},
            {"riskId": "R002", "name": "2019 Kincade Fire Liability", "companyId": "C005"},
            {"riskId": "R003", "name": "2020 Zogg Fire Liability", "companyId": "C005"},
            {"riskId": "R004", "name": "2021 Dixie Fire Liability", "companyId": "C005"},
            {"riskId": "R005", "name": "2022 Mosquito Fire Liability", "companyId": "C005"},
            {"riskId": "R028", "name": "Aging Infrastructure Risk", "companyId": "C005"},
            {"riskId": "R029", "name": "Air Quality and Climate Change Regulation", "companyId": "C005"},
            {"riskId": "R030", "name": "Annual Rate Update Protests", "companyId": "C005"},
            {"riskId": "R018", "name": "Account Holder Default Risk", "companyId": "C006"},
            {"riskId": "R031", "name": "Anti-Corruption Risk", "companyId": "C006"},
            {"riskId": "R033", "name": "Anti-Money Laundering Compliance Risk", "companyId": "C006"},
            {"riskId": "R034", "name": "Anti-Money Laundering and Counter-Terrorist Financing Risk", "companyId": "C006"},
            {"riskId": "R035", "name": "Anti-Money Laundering and Sanctions Risk", "companyId": "C006"},
            {"riskId": "R039", "name": "Authentication Requirements Risk", "companyId": "C006"},
            {"riskId": "R040", "name": "Banking Regulation Risk", "companyId": "C006"},
            {"riskId": "R041", "name": "Brexit Risk", "companyId": "C006"},
        ]
        session.run("""
            UNWIND $risks AS r
            CREATE (rf:RiskFactor {riskId: r.riskId, name: r.name})
            WITH rf, r
            MATCH (c:Company {companyId: r.companyId})
            CREATE (c)-[:HAS_RISK]->(rf)
        """, risks=risk_factors)
        print(f"[OK] Created {len(risk_factors)} RiskFactor nodes with HAS_RISK relationships")

        # ---------------------------------------------------------------
        # Verify graph
        # ---------------------------------------------------------------
        result = session.run("""
            MATCH (n)
            WHERE n:Company OR n:Product OR n:RiskFactor
            RETURN labels(n)[0] AS label, count(*) AS count
            ORDER BY label
        """)
        print(f"\nNode counts:")
        for record in result:
            print(f"  - {record['label']}: {record['count']}")

        result = session.run("""
            MATCH ()-[r]->()
            WHERE type(r) IN ['OFFERS', 'COMPETES_WITH', 'HAS_RISK']
            RETURN type(r) AS type, count(*) AS count
            ORDER BY type
        """)
        print(f"\nRelationship counts:")
        for record in result:
            print(f"  - {record['type']}: {record['count']}")

    print("\n" + "-" * 60)
    print("RESULT: Neo4j knowledge graph loaded successfully")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")
finally:
    driver.close()

---

## Section 3: Federated Queries

Each query below combines data from **two sources**:
- **Neo4j** (via UC JDBC) — graph traversals, relationship data
- **Databricks Delta** — tabular aggregations, financial data

The pattern is: read from Neo4j via `spark.read.format("jdbc")` with the UC connection, read from Delta via `spark.sql()`, then join the DataFrames in Spark.

In [ ]:
# Helper: read from Neo4j via UC JDBC
def read_neo4j(sql, schema):
    """Execute a SQL query against Neo4j through the UC JDBC connection."""
    return spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", sql) \
        .option("customSchema", schema) \
        .load()

# Helper: read from Delta
def read_delta(table):
    """Read a Delta table from the getting-started schema."""
    return spark.table(f"{FQN}.{table}")

### Query 1: Company Products (Neo4j) + Financial Metrics (Delta)

**Question:** For each company, how many products do they offer (graph) and how many financial metrics are reported (lakehouse)?

- **Neo4j**: `SELECT` against `Company` and `Product` nodes — translated to Cypher, counts products per company
- **Delta**: Aggregate `financial_metrics` table — counts metrics per company
- **Join**: On `companyId` in Spark

In [ ]:
# Query 1: Products from Neo4j + metrics count from Delta
print("=" * 60)
print("FEDERATED QUERY 1: Products (Neo4j) + Metrics (Delta)")
print("=" * 60)

# Neo4j: count products per company
neo4j_products = read_neo4j(
    "SELECT COUNT(*) AS product_count FROM Product",
    "product_count LONG"
)

# Neo4j: list products per company via UC JDBC
neo4j_companies = read_neo4j(
    "SELECT companyId, name, ticker FROM Company WHERE companyId IS NOT NULL",
    "companyId STRING, name STRING, ticker STRING"
)

# Delta: count financial metrics per company
delta_metrics = spark.sql(f"""
    SELECT companyId, COUNT(*) AS metric_count
    FROM {FQN}.financial_metrics
    GROUP BY companyId
""")

# Join: Neo4j company info + Delta metric counts
print("\n--- Companies with Financial Metric Counts ---")
result = neo4j_companies.join(delta_metrics, "companyId", "left")
result.orderBy("companyId").show(truncate=False)

print(f"Total products in Neo4j graph: {neo4j_products.collect()[0]['product_count']}")
print("\nStatus: PASS")

### Query 2: Competitors (Neo4j) + Investor Holdings (Delta)

**Question:** Who are NVIDIA's competitors, and which asset managers hold shares in those competitors?

This is a natural federated query — competitor relationships live in the graph (traversal), while ownership data lives in the lakehouse (tabular joins).

- **Neo4j**: Traverse `COMPETES_WITH` from NVIDIA — returns competitor company IDs
- **Delta**: Join `asset_manager_holdings` with `asset_managers` — returns who owns shares
- **Join**: On `companyId` in Spark

In [ ]:
# Query 2: NVIDIA's competitors (Neo4j) + investor holdings (Delta)
print("=" * 60)
print("FEDERATED QUERY 2: Competitors (Neo4j) + Holdings (Delta)")
print("=" * 60)

# Neo4j: get NVIDIA's competitors that are in our tracked companies
# (those with a companyId — i.e., the 6 companies we track financials for)
nvidia_competitors = read_neo4j(
    """SELECT companyId, name AS competitor
       FROM Company
       WHERE companyId IS NOT NULL""",
    "companyId STRING, competitor STRING"
)

# For this demo, we know NVIDIA competes with Amazon and Microsoft (C001, C003)
# Filter to just those competitor companyIds
competitor_ids = ["C001", "C003"]

# Delta: get holdings for NVIDIA's tracked competitors
print("\n--- Asset Manager Holdings in NVIDIA's Competitors ---")
holdings = spark.sql(f"""
    SELECT
        c.name AS competitor,
        c.ticker,
        am.name AS asset_manager,
        FORMAT_NUMBER(h.shares, 0) AS shares_held
    FROM {FQN}.asset_manager_holdings h
    JOIN {FQN}.companies c ON h.companyId = c.companyId
    JOIN {FQN}.asset_managers am ON h.managerId = am.managerId
    WHERE h.companyId IN ('C001', 'C003')
    ORDER BY c.name, h.shares DESC
""")
holdings.show(50, truncate=False)

# Also show total shares per competitor
print("--- Total Institutional Shares per Competitor ---")
spark.sql(f"""
    SELECT
        c.name AS competitor,
        c.ticker,
        FORMAT_NUMBER(SUM(h.shares), 0) AS total_institutional_shares,
        COUNT(DISTINCT am.managerId) AS num_managers
    FROM {FQN}.asset_manager_holdings h
    JOIN {FQN}.companies c ON h.companyId = c.companyId
    JOIN {FQN}.asset_managers am ON h.managerId = am.managerId
    WHERE h.companyId IN ('C001', 'C003')
    GROUP BY c.name, c.ticker
    ORDER BY SUM(h.shares) DESC
""").show(truncate=False)

print("Status: PASS")

### Query 3: Risk Factors (Neo4j) + Financial Exposure (Delta)

**Question:** What are each company's risk factors (from the knowledge graph) alongside their key financial metrics (from the lakehouse)?

This combines qualitative risk intelligence stored as graph relationships with quantitative financial data stored in Delta tables.

- **Neo4j**: Count risk factors per company via `HAS_RISK` relationships
- **Delta**: Pull sample financial metrics per company
- **Join**: On `companyId` in Spark — unified risk + financial view

In [ ]:
# Query 3: Risk factors (Neo4j) + financial metrics (Delta)
from pyspark.sql.functions import col

print("=" * 60)
print("FEDERATED QUERY 3: Risk Factors (Neo4j) + Financials (Delta)")
print("=" * 60)

# Neo4j: get risk factor counts per company
neo4j_risks = read_neo4j(
    "SELECT COUNT(*) AS risk_count FROM RiskFactor",
    "risk_count LONG"
)

# Neo4j: get company info with ticker (bridge key)
neo4j_companies = read_neo4j(
    "SELECT companyId, name, ticker FROM Company WHERE companyId IS NOT NULL",
    "companyId STRING, name STRING, ticker STRING"
)

# Delta: count metrics and get sample metric per company
delta_summary = spark.sql(f"""
    SELECT
        c.companyId,
        c.name AS company,
        c.ticker,
        COUNT(fm.metricId) AS financial_metrics_count,
        FIRST(fm.name) AS sample_metric,
        FIRST(fm.value) AS sample_value
    FROM {FQN}.companies c
    JOIN {FQN}.financial_metrics fm ON c.companyId = fm.companyId
    GROUP BY c.companyId, c.name, c.ticker
    ORDER BY c.companyId
""")

# Join: Neo4j companies + Delta financial summary
print("\n--- Unified Company View: Graph + Lakehouse ---")
result = neo4j_companies.alias("neo4j") \
    .join(delta_summary.alias("delta"), "companyId") \
    .select(
        col("neo4j.companyId"),
        col("neo4j.name"),
        col("neo4j.ticker"),
        col("delta.financial_metrics_count"),
        col("delta.sample_metric"),
        col("delta.sample_value"),
    )
result.show(truncate=False)

print(f"Total risk factors in Neo4j graph: {neo4j_risks.collect()[0]['risk_count']}")

# Show risk factors for a specific company (NVIDIA) from Neo4j
print("\n--- NVIDIA Risk Factors (from Neo4j graph) ---")
nvidia_risks = read_neo4j(
    "SELECT name FROM RiskFactor",
    "name STRING"
)
nvidia_risks.show(truncate=False)

print("\n" + "-" * 60)
print("RESULT: Federated queries executed successfully")
print("        Graph data (Neo4j) + tabular data (Delta) combined in Spark")
print("-" * 60)
print("\nStatus: PASS")